# CustomMobileNetV3Large — Plant Leaf Disease Experiment
**Paper:** A customized MobileNetV3Large-based deep learning framework for plant disease detection (Rahaman et al., 2026)  
**Framework:** TensorFlow 2.x / Keras  
**Dataset:** Hybrid dataset — 26 classes (Bean, Cauliflower, Jackfruit, Malabar Spinach, Mango)

| Scenario | Augmentation |
|----------|--------------|
| Baseline | None (Resize + MobileNetV3 Preprocess) |
| Augmented | RandomFlip(H) + RandomRotation(±20%) + RandomZoom(±30%) + RandomBrightness(±20%) + RandomContrast(±20%) |

In [3]:
# BLOCK 0: Install dependencies
!pip install -q openpyxl Pillow

In [ ]:
# BLOCK 1: Imports
import os
import random
import gc
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
from tensorflow.keras import callbacks as keras_callbacks
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
print(f'TensorFlow: {tf.__version__}')

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print('Seeds set: 42')

strategy = tf.distribute.MirroredStrategy()
n_gpus   = strategy.num_replicas_in_sync
print(f'GPUs available: {n_gpus}')

In [ ]:
# BLOCK 2: Config
DATASET_PATH  = '/kaggle/input/datasets/tminhhi/finaldata/dataset_smartcrop_new'
TRAIN_DIR     = os.path.join(DATASET_PATH, 'train')
VAL_DIR       = os.path.join(DATASET_PATH, 'val')
TEST_DIR      = os.path.join(DATASET_PATH, 'test')
OUTPUT_DIR    = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAPER_CITATION = 'Rahaman et al. (2026) - Discover Artificial Intelligence'
MODEL_NAME     = 'CustomMobileNetV3Large'

IMG_SIZE       = (224, 224)
BATCH_SIZE     = 64
NUM_CLASSES    = len([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])

# Phase 1: Feature Extraction
LR_PHASE1      = 0.001
EPOCHS_PHASE1  = 20

# Phase 2: Fine-Tuning
LR_PHASE2      = 0.0001
EPOCHS_PHASE2  = 15
UNFREEZE_FROM  = 35   # unfreeze backbone layers from index 35 onward

# Callbacks
ES_PATIENCE    = 3
RLROP_FACTOR   = 0.1
RLROP_PATIENCE = 3
RLROP_MIN_LR   = 1e-7

print(f'Config loaded. NUM_CLASSES auto-detected: {NUM_CLASSES}')

In [ ]:
# BLOCK 3: Dataset Exploration + Bar Chart
def count_images(directory):
    counts = {}
    for cls in sorted(os.listdir(directory)):
        cls_path = os.path.join(directory, cls)
        if os.path.isdir(cls_path):
            counts[cls] = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.gif'))
            ])
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts   = count_images(VAL_DIR)
test_counts  = count_images(TEST_DIR)
classes      = list(train_counts.keys())

print(f'Classes detected: {len(classes)}')
print(f'Total  train: {sum(train_counts.values())} | val: {sum(val_counts.values())} | test: {sum(test_counts.values())}')
for c in classes:
    print(f'  {c:45s} train={train_counts.get(c,0):5d}  val={val_counts.get(c,0):5d}  test={test_counts.get(c,0):5d}')

fig, axes = plt.subplots(3, 1, figsize=(18, 18))
for ax, (counts, split, color) in zip(
    axes,
    [(train_counts, 'Train', 'steelblue'),
     (val_counts,   'Val',   'darkorange'),
     (test_counts,  'Test',  'seagreen')]
):
    vals = [counts.get(c, 0) for c in classes]
    bars = ax.bar(range(len(classes)), vals, color=color, edgecolor='black', linewidth=0.4)
    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{split} Set Class Distribution', fontsize=13, fontweight='bold')
    ax.set_ylabel('Image Count')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                str(v), ha='center', va='bottom', fontsize=6)

plt.suptitle('Dataset Statistics — Plant Disease Classes', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/dataset_statistics.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: dataset_statistics.png')

# Class imbalance auto-detection
from sklearn.utils.class_weight import compute_class_weight
counts_array      = np.array([train_counts.get(c, 0) for c in classes])
imbalance_ratio   = counts_array.max() / counts_array.min()
USE_CLASS_WEIGHTS = imbalance_ratio >= 1.5
print(f'\nClass imbalance ratio (max/min): {imbalance_ratio:.2f}')
print(f'→ {"Class weighting ENABLED" if USE_CLASS_WEIGHTS else "Class weighting NOT needed (ratio < 1.5)"}')
if USE_CLASS_WEIGHTS:
    all_labels_flat = []
    for c_idx, c in enumerate(classes):
        all_labels_flat.extend([c_idx] * train_counts.get(c, 0))
    cw = compute_class_weight('balanced', classes=np.arange(len(classes)), y=np.array(all_labels_flat))
    CLASS_WEIGHTS_DICT = {i: float(w) for i, w in enumerate(cw)}
    print(f'Class weights: {CLASS_WEIGHTS_DICT}')
else:
    CLASS_WEIGHTS_DICT = None

In [8]:
# BLOCK 4: Helper Functions (Dataset / Model / Callbacks / Evaluation)

def make_dataset(directory, augment=False, shuffle=True):
    """Load images with optional on-the-fly augmentation via tf.keras layers."""
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        label_mode='int',
        seed=42
    )
    class_names = ds.class_names

    if augment:
        # 4 augmentation techniques as specified in paper
        aug_seq = tf.keras.Sequential([
            layers.RandomFlip('horizontal'),           # Random Horizontal Flip (p=0.5)
            layers.RandomRotation(0.2),                # Random Rotation ±20% of 2pi
            layers.RandomZoom(0.3),                    # Random Zoom ±30%
            layers.RandomBrightness(0.2),              # Brightness adjustment ±20%
            layers.RandomContrast(0.2),                # Contrast adjustment ±20%
        ], name='augmentation_pipeline')

        def aug_preprocess(x, y):
            x = tf.cast(x, tf.float32)
            x = aug_seq(x, training=True)
            x = preprocess_input(x)  # MobileNetV3 normalization
            return x, y
        ds = ds.map(aug_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        def no_aug_preprocess(x, y):
            x = tf.cast(x, tf.float32)
            x = preprocess_input(x)  # MobileNetV3 normalization only
            return x, y
        ds = ds.map(no_aug_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds, class_names


def build_model(num_classes=NUM_CLASSES):
    """
    Custom MobileNetV3Large architecture (Rahaman et al., 2026):
    MobileNetV3Large (ImageNet, frozen) -> GAP -> Dense(256,ReLU) -> Dropout(0.3)
    -> Dense(128,ReLU) -> Dropout(0.2) -> Dense(26, Softmax)
    """
    base_model = MobileNetV3Large(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Phase 1: freeze all backbone layers

    inputs  = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name='CustomMobileNetV3Large')
    return model, base_model


def get_callbacks(ckpt_path):
    return [
        keras_callbacks.EarlyStopping(
            monitor='val_loss', patience=ES_PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        keras_callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=RLROP_FACTOR,
            patience=RLROP_PATIENCE, min_lr=RLROP_MIN_LR,
            verbose=1
        ),
        keras_callbacks.ModelCheckpoint(
            ckpt_path, monitor='val_loss',
            save_best_only=True, verbose=0
        ),
    ]


def get_predictions(model, test_ds):
    """Collect y_true and y_pred over entire test dataset."""
    y_true, y_pred = [], []
    for x_batch, y_batch in test_ds:
        preds = model.predict(x_batch, verbose=0)
        y_true.extend(y_batch.numpy().tolist())
        y_pred.extend(np.argmax(preds, axis=1).tolist())
    return np.array(y_true), np.array(y_pred)


def merge_history(h1, h2):
    """Concatenate Keras History objects from Phase 1 and Phase 2."""
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged


print('Helper functions ready.')

Helper functions ready.


---
## SCENARIO 1: NO AUGMENTATION (Baseline)
**Transform:** Resize 224×224 → MobileNetV3 preprocess_input only

In [9]:
# BLOCK 5: Load Datasets — Scenario 1 (No Augmentation)
print('Loading datasets (No-Aug)...')
train_ds_1, CLASS_NAMES = make_dataset(TRAIN_DIR, augment=False, shuffle=True)
val_ds_1,   _           = make_dataset(VAL_DIR,   augment=False, shuffle=False)
test_ds_1,  _           = make_dataset(TEST_DIR,  augment=False, shuffle=False)

print(f'\nClasses ({len(CLASS_NAMES)}): {CLASS_NAMES}')
print(f'Train batches: {len(train_ds_1)} | Val batches: {len(val_ds_1)} | Test batches: {len(test_ds_1)}')

Loading datasets (No-Aug)...
Found 1597 files belonging to 5 classes.


I0000 00:00:1778258963.415289      56 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778258963.421966      56 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 491 files belonging to 5 classes.
Found 373 files belonging to 5 classes.

Classes (5): ['anthracnose', 'healthy', 'leaf_curl', 'leaf_spot', 'nutrient_deficiency']
Train batches: 25 | Val batches: 8 | Test batches: 6


In [ ]:
# BLOCK 6: Build Model + Train Scenario 1
with strategy.scope():
    model_1, base_model_1 = build_model(NUM_CLASSES)

# Model Summary
model_1.summary()
total_params_1      = model_1.count_params()
trainable_params_1  = sum(tf.size(w).numpy() for w in model_1.trainable_weights)
ntrainable_params_1 = total_params_1 - trainable_params_1
print(f'\nTotal params      : {total_params_1:,}')
print(f'Trainable params  : {trainable_params_1:,}')
print(f'Non-trainable params: {ntrainable_params_1:,}')

# ── Phase 1: Feature Extraction (backbone fully frozen) ─────────────────────
print('\n[Phase 1] Feature Extraction — backbone frozen, LR=0.001, epochs=20')
with strategy.scope():
    model_1.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE1, beta_1=0.9, beta_2=0.999),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
hist_1_p1 = model_1.fit(
    train_ds_1, validation_data=val_ds_1,
    epochs=EPOCHS_PHASE1,
    callbacks=get_callbacks(f'{OUTPUT_DIR}/{MODEL_NAME}_NoAug_Phase1.keras'),
    class_weight=CLASS_WEIGHTS_DICT,
    verbose=1
)

# ── Phase 2: Fine-Tuning (unfreeze backbone from layer 35 onward) ────────────
print('\n[Phase 2] Fine-Tuning — layers[:35] frozen, layers[35:] trainable, LR=0.0001, epochs=15')
for layer in base_model_1.layers[:UNFREEZE_FROM]:
    layer.trainable = False
for layer in base_model_1.layers[UNFREEZE_FROM:]:
    layer.trainable = True

with strategy.scope():
    model_1.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE2, beta_1=0.9, beta_2=0.999),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
hist_1_p2 = model_1.fit(
    train_ds_1, validation_data=val_ds_1,
    epochs=EPOCHS_PHASE2,
    callbacks=get_callbacks(f'{OUTPUT_DIR}/{MODEL_NAME}_NoAug_Best.keras'),
    class_weight=CLASS_WEIGHTS_DICT,
    verbose=1
)

history_1 = merge_history(hist_1_p1, hist_1_p2)
phase1_end_1 = len(hist_1_p1.history['accuracy'])
print('\n[Scenario 1] Training complete.')

In [12]:
# BLOCK 7: Classification Report — Scenario 1
BASELINE_LEARNING_CURVE_PATH   = f'{OUTPUT_DIR}/{MODEL_NAME}_NoAug_LC.png'
BASELINE_CONFUSION_MATRIX_PATH = f'{OUTPUT_DIR}/{MODEL_NAME}_NoAug_CM.png'

y_true_1, y_pred_1 = get_predictions(model_1, test_ds_1)

acc_1    = accuracy_score(y_true_1, y_pred_1)
f1_1     = f1_score(y_true_1, y_pred_1, average='macro', zero_division=0)
recall_1 = recall_score(y_true_1, y_pred_1, average='macro', zero_division=0)

print('=' * 65)
print('[Scenario 1 — No Augmentation] Test Results')
print(f'  Accuracy         : {acc_1:.4f}  ({acc_1 * 100:.2f}%)')
print(f'  F1-Score (Macro) : {f1_1:.4f}')
print(f'  Recall   (Macro) : {recall_1:.4f}')
print(f'  Total Parameters : {model_1.count_params():,}')
print('=' * 65)
print('\nClassification Report (per class):')
print(classification_report(y_true_1, y_pred_1, target_names=CLASS_NAMES, digits=4))

baseline_metrics = {
    'model_name'   : MODEL_NAME,
    'accuracy'     : float(acc_1),
    'f1_macro'     : float(f1_1),
    'recall_macro' : float(recall_1),
    'total_params' : int(model_1.count_params()),
}

[Scenario 1 — No Augmentation] Test Results
  Accuracy         : 0.9357  (93.57%)
  F1-Score (Macro) : 0.9365
  Recall   (Macro) : 0.9357
  Total Parameters : 3,278,618

Classification Report (per class):
                     precision    recall  f1-score   support

        anthracnose     1.0000    1.0000    1.0000        62
            healthy     0.9512    1.0000    0.9750        78
          leaf_curl     1.0000    0.7432    0.8527        74
          leaf_spot     0.9600    0.9474    0.9536        76
nutrient_deficiency     0.8283    0.9880    0.9011        83

           accuracy                         0.9357       373
          macro avg     0.9479    0.9357    0.9365       373
       weighted avg     0.9434    0.9357    0.9341       373



In [13]:
# BLOCK 8: Confusion Matrix — Scenario 1 (CELL RIÊNG BIỆT)
cm_1   = confusion_matrix(y_true_1, y_pred_1)
disp_1 = ConfusionMatrixDisplay(confusion_matrix=cm_1, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(16, 14))
disp_1.plot(cmap='Blues', ax=ax, xticks_rotation=45, colorbar=True)
ax.set_title(f'Confusion Matrix — {MODEL_NAME} (No-Aug)', fontsize=16, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
plt.tight_layout()
plt.savefig(BASELINE_CONFUSION_MATRIX_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {BASELINE_CONFUSION_MATRIX_PATH}')

Saved: /kaggle/working/CustomMobileNetV3Large_NoAug_CM.png


In [ ]:
# BLOCK 9: Learning Curve — Scenario 1 (CELL RIÊNG BIỆT)
epochs_range_1 = range(1, len(history_1['accuracy']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range_1, history_1['accuracy'],     label='Train Accuracy', color='royalblue', linewidth=2)
axes[0].plot(epochs_range_1, history_1['val_accuracy'], label='Val Accuracy',   color='darkorange', linewidth=2, linestyle='--')
axes[0].axvline(x=phase1_end_1, color='red', linestyle=':', linewidth=1.5, label='Phase 1 → 2')
axes[0].set_title('Training & Validation Accuracy', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range_1, history_1['loss'],     label='Train Loss', color='royalblue', linewidth=2)
axes[1].plot(epochs_range_1, history_1['val_loss'], label='Val Loss',   color='darkorange', linewidth=2, linestyle='--')
axes[1].axvline(x=phase1_end_1, color='red', linestyle=':', linewidth=1.5, label='Phase 1 → 2')
axes[1].set_title('Training & Validation Loss', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cross Entropy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Learning Curve — {MODEL_NAME} (No-Aug)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(BASELINE_LEARNING_CURVE_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {BASELINE_LEARNING_CURVE_PATH}')

# Free GPU memory before Scenario 2
del model_1, base_model_1
tf.keras.backend.clear_session()
gc.collect()
print('[Memory] GPU memory cleared before Scenario 2.')

---
## SCENARIO 2: WITH AUGMENTATION
**Augmentations:** RandomFlip(H) + RandomRotation(±20%) + RandomZoom(±30%) + RandomBrightness(±20%) + RandomContrast(±20%)

In [15]:
# BLOCK 10: Load Datasets — Scenario 2 (With Augmentation)
print('Loading datasets (Aug)...')
train_ds_2, _ = make_dataset(TRAIN_DIR, augment=True,  shuffle=True)
val_ds_2,   _ = make_dataset(VAL_DIR,   augment=False, shuffle=False)  # no aug on val
test_ds_2,  _ = make_dataset(TEST_DIR,  augment=False, shuffle=False)  # no aug on test

print(f'Train batches: {len(train_ds_2)} | Val batches: {len(val_ds_2)} | Test batches: {len(test_ds_2)}')
print('Aug pipeline: RandomFlip(H) + RandomRotation(0.2) + RandomZoom(0.3) + RandomBrightness(0.2) + RandomContrast(0.2)')

Loading datasets (Aug)...
Found 1597 files belonging to 5 classes.
Found 491 files belonging to 5 classes.
Found 373 files belonging to 5 classes.
Train batches: 25 | Val batches: 8 | Test batches: 6
Aug pipeline: RandomFlip(H) + RandomRotation(0.2) + RandomZoom(0.3) + RandomBrightness(0.2) + RandomContrast(0.2)


In [ ]:
# BLOCK 11: Build Fresh Model + Train Scenario 2 (no data leakage from Scenario 1)
with strategy.scope():
    model_2, base_model_2 = build_model(NUM_CLASSES)

# Model Summary
model_2.summary()
total_params_2      = model_2.count_params()
trainable_params_2  = sum(tf.size(w).numpy() for w in model_2.trainable_weights)
ntrainable_params_2 = total_params_2 - trainable_params_2
print(f'\nTotal params      : {total_params_2:,}')
print(f'Trainable params  : {trainable_params_2:,}')
print(f'Non-trainable params: {ntrainable_params_2:,}')

# ── Phase 1: Feature Extraction (backbone fully frozen) ─────────────────────
print('\n[Phase 1] Feature Extraction — backbone frozen, LR=0.001, epochs=20')
with strategy.scope():
    model_2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE1, beta_1=0.9, beta_2=0.999),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
hist_2_p1 = model_2.fit(
    train_ds_2, validation_data=val_ds_2,
    epochs=EPOCHS_PHASE1,
    callbacks=get_callbacks(f'{OUTPUT_DIR}/{MODEL_NAME}_Aug_Phase1.keras'),
    class_weight=CLASS_WEIGHTS_DICT,
    verbose=1
)

# ── Phase 2: Fine-Tuning (unfreeze backbone from layer 35 onward) ────────────
print('\n[Phase 2] Fine-Tuning — layers[:35] frozen, layers[35:] trainable, LR=0.0001, epochs=15')
for layer in base_model_2.layers[:UNFREEZE_FROM]:
    layer.trainable = False
for layer in base_model_2.layers[UNFREEZE_FROM:]:
    layer.trainable = True

with strategy.scope():
    model_2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_PHASE2, beta_1=0.9, beta_2=0.999),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
hist_2_p2 = model_2.fit(
    train_ds_2, validation_data=val_ds_2,
    epochs=EPOCHS_PHASE2,
    callbacks=get_callbacks(f'{OUTPUT_DIR}/{MODEL_NAME}_Aug_Best.keras'),
    class_weight=CLASS_WEIGHTS_DICT,
    verbose=1
)

history_2 = merge_history(hist_2_p1, hist_2_p2)
phase1_end_2 = len(hist_2_p1.history['accuracy'])
print('\n[Scenario 2] Training complete.')

In [18]:
# BLOCK 12: Classification Report — Scenario 2
AUG_LEARNING_CURVE_PATH   = f'{OUTPUT_DIR}/{MODEL_NAME}_Aug_LC.png'
AUG_CONFUSION_MATRIX_PATH = f'{OUTPUT_DIR}/{MODEL_NAME}_Aug_CM.png'

y_true_2, y_pred_2 = get_predictions(model_2, test_ds_2)

acc_2    = accuracy_score(y_true_2, y_pred_2)
f1_2     = f1_score(y_true_2, y_pred_2, average='macro', zero_division=0)
recall_2 = recall_score(y_true_2, y_pred_2, average='macro', zero_division=0)

print('=' * 65)
print('[Scenario 2 — Augmented] Test Results')
print(f'  Accuracy         : {acc_2:.4f}  ({acc_2 * 100:.2f}%)')
print(f'  F1-Score (Macro) : {f1_2:.4f}')
print(f'  Recall   (Macro) : {recall_2:.4f}')
print(f'  Total Parameters : {model_2.count_params():,}')
print('=' * 65)
print('\nClassification Report (per class):')
print(classification_report(y_true_2, y_pred_2, target_names=CLASS_NAMES, digits=4))

aug_metrics = {
    'model_name'   : MODEL_NAME,
    'accuracy'     : float(acc_2),
    'f1_macro'     : float(f1_2),
    'recall_macro' : float(recall_2),
    'total_params' : int(model_2.count_params()),
}

[Scenario 2 — Augmented] Test Results
  Accuracy         : 0.8686  (86.86%)
  F1-Score (Macro) : 0.8694
  Recall   (Macro) : 0.8677
  Total Parameters : 3,278,618

Classification Report (per class):
                     precision    recall  f1-score   support

        anthracnose     1.0000    0.9194    0.9580        62
            healthy     0.8837    0.9744    0.9268        78
          leaf_curl     1.0000    0.6081    0.7563        74
          leaf_spot     0.8750    0.9211    0.8974        76
nutrient_deficiency     0.7238    0.9157    0.8085        83

           accuracy                         0.8686       373
          macro avg     0.8965    0.8677    0.8694       373
       weighted avg     0.8888    0.8686    0.8659       373



In [19]:
# BLOCK 13: Confusion Matrix — Scenario 2 (CELL RIÊNG BIỆT)
cm_2   = confusion_matrix(y_true_2, y_pred_2)
disp_2 = ConfusionMatrixDisplay(confusion_matrix=cm_2, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(16, 14))
disp_2.plot(cmap='Blues', ax=ax, xticks_rotation=45, colorbar=True)
ax.set_title(f'Confusion Matrix — {MODEL_NAME} (Aug)', fontsize=16, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
plt.tight_layout()
plt.savefig(AUG_CONFUSION_MATRIX_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {AUG_CONFUSION_MATRIX_PATH}')

Saved: /kaggle/working/CustomMobileNetV3Large_Aug_CM.png


In [20]:
# BLOCK 14: Learning Curve — Scenario 2 (CELL RIÊNG BIỆT)
epochs_range_2 = range(1, len(history_2['accuracy']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range_2, history_2['accuracy'],     label='Train Accuracy', color='royalblue', linewidth=2)
axes[0].plot(epochs_range_2, history_2['val_accuracy'], label='Val Accuracy',   color='darkorange', linewidth=2, linestyle='--')
axes[0].axvline(x=phase1_end_2, color='red', linestyle=':', linewidth=1.5, label='Phase 1 → 2')
axes[0].set_title('Training & Validation Accuracy', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range_2, history_2['loss'],     label='Train Loss', color='royalblue', linewidth=2)
axes[1].plot(epochs_range_2, history_2['val_loss'], label='Val Loss',   color='darkorange', linewidth=2, linestyle='--')
axes[1].axvline(x=phase1_end_2, color='red', linestyle=':', linewidth=1.5, label='Phase 1 → 2')
axes[1].set_title('Training & Validation Loss', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Cross Entropy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'Learning Curve — {MODEL_NAME} (Aug)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(AUG_LEARNING_CURVE_PATH, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {AUG_LEARNING_CURVE_PATH}')

Saved: /kaggle/working/CustomMobileNetV3Large_Aug_LC.png


---
## EXCEL EXPORT
Xuất `results_summary.xlsx` với 2 sheet (Baseline + Augmented), ảnh Learning Curve và Confusion Matrix nhúng trực tiếp.

In [21]:
# BLOCK 15: Export results_summary.xlsx
import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image as XLImage
from openpyxl.utils import get_column_letter

EXCEL_OUTPUT_PATH = '/kaggle/working/results_summary.xlsx'
PAPER_CITATION_XL = 'Rahaman et al. (2026)'

HEADERS    = ['STT', 'Ten Model', 'Accuracy', 'F1-score', 'Recall',
              'Learning Curve', 'Confusion Matrix', 'Parameters']
COL_WIDTHS = [6, 40, 12, 12, 12, 38, 38, 18]
IMG_ROW_HEIGHT = 200


def style_header_row(ws):
    fill   = PatternFill('solid', fgColor='1F4E79')
    font   = Font(bold=True, color='FFFFFF', size=11)
    border = Border(
        bottom=Side(style='medium', color='FFFFFF'),
        right=Side(style='thin',   color='FFFFFF'),
    )
    for col, header in enumerate(HEADERS, start=1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.fill, cell.font, cell.border = fill, font, border
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)


def write_result_row(ws, row_num, stt, metrics, lc_path, cm_path):
    data = [
        stt,
        f"{metrics['model_name']} ({PAPER_CITATION_XL})",
        round(metrics['accuracy'],     4),
        round(metrics['f1_macro'],     4),
        round(metrics['recall_macro'], 4),
        '',   # Learning Curve placeholder
        '',   # Confusion Matrix placeholder
        f"{metrics['total_params']:,}",
    ]
    for col, val in enumerate(data, start=1):
        cell = ws.cell(row=row_num, column=col, value=val)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    ws.row_dimensions[row_num].height = IMG_ROW_HEIGHT

    if os.path.exists(lc_path):
        img_lc = XLImage(lc_path)
        img_lc.width, img_lc.height = 280, 190
        ws.add_image(img_lc, f'F{row_num}')

    if os.path.exists(cm_path):
        img_cm = XLImage(cm_path)
        img_cm.width, img_cm.height = 280, 190
        ws.add_image(img_cm, f'G{row_num}')


def build_sheet(wb, sheet_name, metrics, lc_path, cm_path):
    ws = wb.create_sheet(title=sheet_name)
    for col, width in enumerate(COL_WIDTHS, start=1):
        ws.column_dimensions[get_column_letter(col)].width = width
    ws.row_dimensions[1].height = 30
    style_header_row(ws)
    write_result_row(ws, row_num=2, stt=1, metrics=metrics,
                     lc_path=lc_path, cm_path=cm_path)
    return ws


# Build workbook
wb = Workbook()
wb.remove(wb.active)

build_sheet(wb, 'Baseline',
            metrics=baseline_metrics,
            lc_path=BASELINE_LEARNING_CURVE_PATH,
            cm_path=BASELINE_CONFUSION_MATRIX_PATH)

build_sheet(wb, 'Augmented',
            metrics=aug_metrics,
            lc_path=AUG_LEARNING_CURVE_PATH,
            cm_path=AUG_CONFUSION_MATRIX_PATH)

wb.save(EXCEL_OUTPUT_PATH)
print(f'[Excel] Saved: {EXCEL_OUTPUT_PATH}')
print(f'  Baseline  -> Accuracy={baseline_metrics["accuracy"]:.4f} | F1={baseline_metrics["f1_macro"]:.4f} | Recall={baseline_metrics["recall_macro"]:.4f} | Params={baseline_metrics["total_params"]:,}')
print(f'  Augmented -> Accuracy={aug_metrics["accuracy"]:.4f}      | F1={aug_metrics["f1_macro"]:.4f}      | Recall={aug_metrics["recall_macro"]:.4f}      | Params={aug_metrics["total_params"]:,}')

[Excel] Saved: /kaggle/working/results_summary.xlsx
  Baseline  -> Accuracy=0.9357 | F1=0.9365 | Recall=0.9357 | Params=3,278,618
  Augmented -> Accuracy=0.8686      | F1=0.8694      | Recall=0.8677      | Params=3,278,618
